# HeartTwin KIT chamber-law development reassessment

Run in the user's Edge Colab CPU runtime. This notebook is a new rerun recipe, not an executed result artifact. All outer data were previously seen; one volunteer, no independent or clinical validation. KIT v1.1: https://zenodo.org/records/10526554, CC BY-NC 4.0. Keep the original failed benchmark unchanged. Model phase changes do not alter measured timestamps.

Run cells through the export in order for a fresh rerun. The final **rescue cell** is separate and disabled by default: use it only to recover the original live runtime without rerunning experiments. Numerical runs can take several minutes per family.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
PIN = 'ea7cafa7328900cfd4ae02d49e73a3f63faced20'
ROOT = Path('/content/HeartTwin_Chamber_Refinement_' + PIN[:12])
ROOT.mkdir(exist_ok=True)
LOGS = ROOT/'setup-logs'; LOGS.mkdir(exist_ok=True)
ENV = dict(os.environ, MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
def run(command, name, timeout=2400, cwd=None):
    print('Starting', name, flush=True)
    log_path = LOGS/(name+'.log')
    with log_path.open('w') as log:
        result = subprocess.run(list(map(str, command)), cwd=cwd, env=ENV,
            stdout=log, stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        print(log_path.read_text()[-10000:])
        raise RuntimeError(name+' failed; retain the log')
PINS = {
 'Virelion-HeartTwin': PIN,
 'Virelion-CardiMech': '5dc14d886ccbda5a1398b863f9ae7821bbffa18b',
 'Virelion-CardiFlow': '670d370635ba09b2908f7e6736299719e86fc36b',
 'Virelion-CardiInfer': 'b4f0c3c30da5986c6a98dba6ceccf1e1de95c881',
 'Virelion-CardiEP': 'f9bfcb78cdb857e5995ce244d8645f7370d6b3bb',
 'Virelion-CardiAnatomy': 'e3cc3450c49b84334dd2da696f33a75f4f61557f',
 'Virelion-ElectroTrace': 'a4cf37ad2e1ed6719a23befa2be5e47af4b4d1de'}
for name, pin in PINS.items():
    path = ROOT/name
    if not path.exists():
        run(['git','init',path], 'init-'+name)
        run(['git','-C',path,'fetch','--depth','1',
             'https://github.com/Virelion-Biotech/'+name+'.git',pin], 'fetch-'+name)
        run(['git','-C',path,'checkout','--detach','FETCH_HEAD'], 'checkout-'+name)
    assert subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True).strip()==pin
    assert not subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=no'],text=True).strip()
venv = ROOT/'venv'
if not venv.exists():
    run([sys.executable,'-m','venv','--without-pip',venv], 'venv')
python = venv/'bin/python'
ENV['PATH'] = str(python.parent)+os.pathsep+ENV['PATH']
run([sys.executable,'-m','pip','--python',python,'install',
 'pip==25.1.1','numpy==2.2.6','scipy==1.16.3','pydantic==2.12.3',
 'PyYAML==6.0.3','pytest==8.4.2','ruff==0.14.1','meshio==5.3.5',
 'nibabel==5.3.2','pydicom==3.0.1','jsonschema==4.25.1',
 'matplotlib==3.10.7','hypothesis>=6.100','flask>=3.0'], 'dependencies')
for name in PINS:
    run([python,'-m','pip','install','-e',ROOT/name], 'install-'+name)
run([python,'-m','pip','check'], 'pip-check')
run([python,'-m','pip','freeze'], 'environment')
(LOGS/'pins.json').write_text(json.dumps(PINS,indent=2))
repo = ROOT/'Virelion-HeartTwin'
baseline = repo/'validation/kit_edgar_2026_10_20'
FAMILIES = [
 ('constitutive-first','results_chamber_laws','diagnose_kit_chamber_laws.py',[]),
 ('bound-refinement','results_chamber_refinement','diagnose_kit_chamber_laws.py',['--refine-calibration']),
 ('profile-phase','results_stiffness_profile','profile_kit_passive_stiffness.py',[]),
 ('profile-fixed-phase','results_stiffness_fixed_phase','profile_kit_passive_stiffness.py',['--fixed-phase'])]
SELECTION = ROOT/'results_chamber_final'; SELECTION.mkdir(exist_ok=True)
print('Pinned environment ready')


## Four recorded families

Fit first 17 original calibration samples; choose within each family using the next six; refit all 23. Freeze predictions before reading archived outer values inside each script. This retrospective reproduction runs all families sequentially; it does not recreate a prospective unseen-data experiment. Preserve every family, including controls with lower outer error that lose the inner selection criterion.


In [ ]:
for name, folder, script, extra in FAMILIES:
    run([python,repo/'scripts'/script,'--baseline',baseline,
         '--output',ROOT/folder,*extra], name, cwd=repo)
    print(name, 'complete', flush=True)


In [ ]:
import shutil
# Cross-family choice reads only inner selection files.
ranking = []
for name, folder, _, _ in FAMILIES:
    selected = json.loads((ROOT/folder/'selection.json').read_text())
    ranking.append({'family':name,'folder':folder,'inner_rmse_mL':selected['inner_rmse_mL']})
ranking.sort(key=lambda item:(item['inner_rmse_mL'],item['family']))
winner = ranking[0]
choice = {'rule':'Lowest inner calibration RMSE; no outer-score selection',
          'prior_outer_values_seen':True,'independent_validation':False,
          'ranking':ranking,'selected_family':winner['family']}
(SELECTION/'cross_round_selection.json').write_text(json.dumps(choice,indent=2))
selected_dir = ROOT/winner['folder']
for source, target in [('results.json','selected_results.json'),
                       ('frozen_request.json','selected_request.json'),
                       ('frozen_prediction.json','selected_prediction.json')]:
    shutil.copyfile(selected_dir/source,SELECTION/target)
choice['prediction_sha256'] = hashlib.sha256((SELECTION/'selected_prediction.json').read_bytes()).hexdigest()
(SELECTION/'cross_round_selection.json').write_text(json.dumps(choice,indent=2))
print(json.dumps(choice,indent=2))
print((SELECTION/'selected_results.json').read_text())


## Software checks

These checks establish software behavior only. The chamber suite includes a real CardiInfer synthetic inverse/replay test. No patient pressure identification, RV improvement, independent patient validation, spatial mechanics or EDGAR electrical validation is inferred from a pass.


In [ ]:
run([python,repo/'scripts/validate_patient_lifecycle.py','--output',SELECTION/'lifecycle'],
    'lifecycle',cwd=repo)
run([python,'-m','ruff','check','hearttwin/patient_closed_loop.py',
     'hearttwin/patient_domain.py','tests/test_patient_chamber_laws.py',
     'scripts/diagnose_kit_chamber_laws.py','scripts/profile_kit_passive_stiffness.py'],
    'changed-file-lint',cwd=repo)
print((SELECTION/'lifecycle/STATUS.json').read_text())


## Export this rerun

The ZIP contains generated results and logs, with a SHA256 manifest. This notebook does not reconstruct the prior session's independent solver script or its figure; recover those original files using the rescue cell below. The rerun uses fresh fits and may differ with environment or solver changes; compare the recorded pins and logs.


In [ ]:
def export_results(root, folders, output_name):
    paths = sorted(p for folder in folders for p in (root/folder).rglob('*') if p.is_file())
    if not paths:
        raise RuntimeError('No result files found; do not create an empty evidence bundle')
    manifest = {str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
    manifest_path = root/(output_name+'_manifest.json')
    manifest_path.write_text(json.dumps(manifest,indent=2))
    archive = root/(output_name+'.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zipped:
        for p in paths+[manifest_path]:
            zipped.write(p,str(p.relative_to(root)))
    print(json.dumps({'archive':str(archive),'bytes':archive.stat().st_size,
       'sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'file_count':len(paths)},indent=2))
    return archive
archive = export_results(ROOT,[folder for _,folder,_,_ in FAMILIES]+
                         ['results_chamber_final','setup-logs'],'KIT_chamber_refinement_rerun')
from google.colab import files
files.download(str(archive))


## Rescue only: original live runtime

Leave RUN_RESCUE=False during a fresh rerun. To recover the existing session, copy this cell alone into the original live notebook, set RUN_RESCUE=True, and run it. It reads existing files only and does not launch numerical work. Do not restart or reload that runtime. A running final process is reported without waiting; export again once it finishes. Keep all failures and logs. Review before redistribution; no source geometry folder is included.


In [ ]:
RUN_RESCUE = False
if RUN_RESCUE:
    from pathlib import Path
    import hashlib, json, zipfile
    rescue_root = Path('/content/HeartTwin_Diagnosis_20261020')
    rescue_folders = ['results_chamber_laws','results_chamber_refinement',
        'results_stiffness_profile','results_stiffness_fixed_phase','results_chamber_final']
    if not rescue_root.exists():
        raise RuntimeError('Original runtime directory unavailable; do not fabricate its evidence')
    if 'final_life_process' in globals():
        print('FINAL_LIFECYCLE_EXIT',final_life_process.poll())
    for relative in ['results_chamber_final/lifecycle.log',
                     'results_chamber_final/lifecycle/STATUS.json',
                     'results_chamber_final/lifecycle/patient-and-workbench-tests.log']:
        rescue_path = rescue_root/relative
        if rescue_path.exists():
            print(relative, rescue_path.read_text()[-5000:])
    rescue_paths = sorted(p for folder in rescue_folders
        for p in (rescue_root/folder).rglob('*') if p.is_file())
    if not rescue_paths:
        raise RuntimeError('Original result files missing')
    rescue_manifest = {str(p.relative_to(rescue_root)):hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in rescue_paths}
    manifest_path = rescue_root/'KIT_chamber_original_manifest.json'
    manifest_path.write_text(json.dumps(rescue_manifest,indent=2))
    rescue_archive = rescue_root/'KIT_chamber_original_results.zip'
    with zipfile.ZipFile(rescue_archive,'w',zipfile.ZIP_DEFLATED) as zipped:
        for p in rescue_paths+[manifest_path]:
            zipped.write(p,str(p.relative_to(rescue_root)))
    print('ARCHIVE_SHA256',hashlib.sha256(rescue_archive.read_bytes()).hexdigest())
    print('ARCHIVE_BYTES',rescue_archive.stat().st_size,'FILES',len(rescue_paths))
    from google.colab import files
    files.download(str(rescue_archive))
